# 3. Batch correction with ComBat on the integrated dataset

After normalization and per-sample preprocessing, the next step is to correct for **batch effects** across the 8 embryonic mouse brain samples (4 WT and 4 KO). Even after careful QC and normalization, technical differences between samples (for example, library preparation, sequencing run, or handling) can introduce systematic shifts in expression that obscure biological signals.

In this notebook, we:

- load a combined AnnData object containing all 8 samples and a chosen normalized layer,
- select a subset of highly expressed genes (as a proxy for highly variable genes) in a memory-efficient way,
- compute a low-dimensional PCA representation using TruncatedSVD,
- apply **ComBat** batch correction across samples, and
- assess clustering quality using Leiden clustering and silhouette scores on the corrected embedding.

The output is a ComBat-corrected `.h5ad` file that will be used for downstream visualization, clustering, and differential expression analyses across batches.

## 3.1. Imports and global settings

In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import anndata

from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import silhouette_score
from matplotlib import pyplot as plt

# Scanpy verbosity and plotting
sc.settings.verbosity = 0
sc.settings.set_figure_params(dpi=80, facecolor="white", frameon=False)

## 3.2. Configuration

We specify the combined input file (if you have multiple files after QC, use the `concatenation.ipynb` file to concatenate you datasets), the normalization layer to use for ComBat, and the set of Leiden resolutions at which we will evaluate clustering via silhouette scores.

In [ ]:
# Combined object with multiple normalization layers
INPUT_FILE = "combined_8_samples_all_layers_outer.h5ad"

# Normalization layer to use for ComBat
NORM_LAYER = "scran_normalization" # use whichever layer you want

# Leiden resolutions to scan when computing silhouette scores
LEIDEN_RESOLUTIONS = [0.25, 0.5, 0.7, 1.0, 2.0]

print(f"Using input file: {INPUT_FILE}")
print(f"Normalization layer for ComBat: {NORM_LAYER}")

## 3.3. Motivation and strategy

When multiple single-cell RNA-seq samples are combined, technical variability between samples (batch effects) can create artificial separation or mixing of cells that is unrelated to the biological condition of interest. For the wild-type vs knockout dataset, we want cell states and types to align across samples so that differences we observe are driven primarily by genotype and biology rather than technical artifacts.

Here we use **ComBat**, a well-established batch correction method originally developed for microarrays and extended to RNA-seq. ComBat models batch-specific effects and adjusts expression values accordingly, while attempting to preserve biological variation.

Because the combined dataset is relatively large, we adopt a memory-aware strategy:

1. Load the combined `.h5ad` file in backed mode.
2. From the chosen normalization layer (here `scran_normalization`), compute the mean expression per gene and select the top 10,000 genes as a proxy for highly variable genes.
3. Load only these 10,000 genes into memory to create a smaller AnnData object.
4. Run PCA (via `TruncatedSVD` for sparse efficiency), then compute neighbors and UMAP **before** ComBat.
5. Apply ComBat using the `batch` column in `.obs` to specify sample identity.
6. Recompute PCA, neighbors, UMAP, and Leiden clustering **after** ComBat.
7. Evaluate clustering at several resolutions using the silhouette score and save the corrected object for downstream analyses.

## 3.4. Loading a 10k-gene subset for ComBat

To keep memory usage manageable, we do not operate on all genes. Instead, we select the top 10,000 genes by mean expression in the chosen normalization layer and build a compact AnnData object that contains:

- the selected genes from `NORM_LAYER`, and
- the full cell-level metadata and embeddings from the combined object.

In [ ]:
def load_for_combat(input_file: str, norm_layer_name: str) -> anndata.AnnData:
    """Load combined data and keep only the top 10k genes by mean expression in a given layer."""
    print(f"Loading 10k genes from layer '{norm_layer_name}' (memory-efficient)...")

    # Load in backed mode to avoid reading everything into memory
    adata_full = sc.read_h5ad(input_file, backed="r")

    # Top 10k genes by mean expression as a fast HVG proxy
    layer_means = np.array(adata_full.layers[norm_layer_name].mean(axis=0)).flatten()
    hvg_idx = np.argsort(layer_means)[-10000:]
    hvg_genes = adata_full.var_names[hvg_idx]

    # Load only these 10k genes
    X_hvg = adata_full.layers[norm_layer_name][:, hvg_idx]

    # Build a new AnnData object in memory
    adata = anndata.AnnData(
        X=X_hvg,
        obs=adata_full.obs.copy(),
        var=adata_full.var.loc[hvg_genes].copy(),
        obsm=adata_full.obsm.copy() if "obsm" in adata_full.__dict__ else {},
    )
    print(f"Loaded subset: {adata.n_obs} cells × {adata.n_vars} genes")

    adata_full.file.close()
    return adata

## 3.5. Clustering and silhouette evaluation

To evaluate how well ComBat has harmonized batches while preserving biological structure, we run Leiden clustering at multiple resolutions and compute silhouette scores based on the PCA embedding. Higher silhouette scores indicate more compact and well-separated clusters.

In [ ]:
def run_leiden_silhouette(adata: anndata.AnnData,
                          embedding_key: str,
                          label_prefix: str):
    """Run Leiden at multiple resolutions and compute silhouette scores."""
    sil_scores = {}
    for res in LEIDEN_RESOLUTIONS:
        r_str = str(res).replace(".", "_")
        leiden_key = f"{label_prefix}_{r_str}"
        sc.tl.leiden(adata, resolution=res, key_added=leiden_key)
        labels = adata.obs[leiden_key].astype(str)
        score = silhouette_score(adata.obsm[embedding_key], labels)
        sil_scores[res] = score
        print(f"Leiden res={res}: silhouette = {score:.4f}")

    best_res = max(sil_scores, key=sil_scores.get)
    best_key = f"{label_prefix}_{str(best_res).replace('.', '_')}"
    print(f"Best resolution for {label_prefix}: {best_res}")
    return best_res, best_key, sil_scores

## 3.6. ComBat batch correction pipeline

The core ComBat pipeline proceeds as follows:

1. Compute PCA on the selected 10k genes using `TruncatedSVD` (suitable for large, possibly sparse matrices).
2. Build a neighbor graph and compute UMAP **before** ComBat; store these coordinates.
3. Convert the expression matrix to dense if necessary, since Scanpy’s ComBat implementation expects dense input.
4. Run `sc.pp.combat` using the `batch` column in `.obs` to specify sample/batch identity.
5. Recompute PCA, neighbors, and UMAP **after** ComBat, and store the corrected embedding.
6. Run Leiden clustering and compute silhouette scores on the corrected PCA space.
7. Save the ComBat-corrected AnnData object for further analysis.

In [ ]:
def run_combat_pipeline(adata: anndata.AnnData,
                        norm_name: str):
    """
    ComBat batch correction + metrics + before/after UMAP.
    Uses Scanpy's ComBat implementation.
    """

    # PCA / neighbors / UMAP BEFORE ComBat
    print("Running PCA on HVGs with TruncatedSVD...")
    svd = TruncatedSVD(n_components=50, algorithm="arpack")
    adata.obsm["X_pca"] = svd.fit_transform(adata.X)
    print(f"TruncatedSVD complete: {svd.explained_variance_ratio_.sum():.3f} variance explained")

    print("Computing neighbors and UMAP BEFORE ComBat...")
    sc.pp.neighbors(adata, use_rep="X_pca", n_neighbors=15, n_pcs=40)
    sc.tl.umap(adata, random_state=0)
    adata.obsm["X_umap_before"] = adata.obsm["X_umap"].copy()

    # ComBat expects dense arrays in Scanpy implementation
    print("Running ComBat batch correction...")
    if sp.issparse(adata.X):
        adata.X = adata.X.toarray()
    sc.pp.combat(adata, key="batch")

    # PCA / neighbors / UMAP AFTER ComBat
    print("Computing PCA, neighbors, and UMAP AFTER ComBat...")
    sc.pp.pca(adata, n_comps=50, svd_solver="arpack")
    sc.pp.neighbors(adata, use_rep="X_pca", n_neighbors=15, n_pcs=40)
    sc.tl.umap(adata, random_state=0)
    adata.obsm["X_umap_after"] = adata.obsm["X_umap"].copy()

    # Leiden + silhouette on corrected PCA embedding
    print("Running Leiden + silhouette on ComBat PCs...")
    best_res, best_leiden_key, sil_scores = run_leiden_silhouette(
        adata,
        embedding_key="X_pca",
        label_prefix=f"leiden_combat_{norm_name}",
    )

    # Save corrected object
    out_file = f"combined_8_samples_batch_corrected_combat_{norm_name}.h5ad"
    adata.write(out_file)
    print(f"Saved ComBat-corrected data to: {out_file}")

    return {
        "norm_layer": norm_name,
        "best_res": best_res,
        "best_leiden_key": best_leiden_key,
        "sil_scores": sil_scores,
        "outfile": out_file,
    }

## 3.7. Run ComBat on the chosen normalization layer

We now apply the ComBat pipeline to a single normalization layer (here, analytic Pearson residuals). The same notebook can be adapted to loop over multiple layers and compare their batch-correction performance.

In [ ]:
# Inspect available layers in the combined file
print("Available layers in input file:")
adata_test = sc.read_h5ad(INPUT_FILE, backed="r")
print(list(adata_test.layers.keys()))
adata_test.file.close()

# Load and prepare data for one normalization layer with ~10k genes
print(f"\nLoading data for normalization layer: {NORM_LAYER}")
adata = load_for_combat(INPUT_FILE, NORM_LAYER)

# Run ComBat pipeline
stats = run_combat_pipeline(adata, NORM_LAYER)

# Optional: explicit PCA before/after using separate SVD objects
svd_before = TruncatedSVD(n_components=50, algorithm="arpack")
adata.obsm["X_pca_before_recheck"] = svd_before.fit_transform(adata.X)

svd_after = TruncatedSVD(n_components=50, algorithm="arpack")
adata.obsm["X_pca_after_recheck"] = svd_after.fit_transform(adata.X)

print("ComBat batch correction completed.")